In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Saving the Goat

An agricultural economist has to sell a *wolf*, a *goat*, and a *cabbage* on a market place.  In order to
reach the market place, she has to cross a river.  The boat that she can use is so small that it can
only accommodate either the goat, the wolf, or the cabbage in addition to the agricultural economist.
Now if the agricultural economist leaves the wolf alone with the goat, the wolf will eat the goat.
If, instead, the agricultural economist leaves the goat with the cabbage, the goat will eat the cabbage.
Is it possible for the agricultural economist to develop a schedule that allows her to cross the river
without either the goat or the cabbage being eaten?

We will encode this problem as a *symbolic transition system* and then solve it with the help of the algorithm of Davis and Putnam.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossings of the river.  We use the following variables:
* $\texttt{F}i$ for $i\in\{0,\cdots,n\}$ is `true` if the farmer is on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{W}i$ for $i\in\{0,\cdots,n\}$ is `true` if the wolf is on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{G}i$ for $i\in\{0,\cdots,n\}$ is `true` if the goat is on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is `true` if the cabbage is on the western shore after the 
  $i^{\textrm{th}}$ crossing.

We import the notebook `07-Davis-Putnam-JW.ipynb`.  This notebook imports the notebook `04-CNF.ipynb`, which implements the function `normalize` that takes a formula and transforms it into a set of clauses.  The notebook `04-CNF.ipynb` in turn imports the parser for propositional logic.

In [2]:
importNotebook "07-Davis-Putnam-JW.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb, 07-Davis-Putnam-JW.ipynb

Using the parser and the function `normalize` we can implement the function `parseKNF`.  It takes one argument:
- `s` is a string representing a propositional formula.

It returns a set of clauses that is equivalent to the formula represented by `s`.

In [3]:
let parseKNF (s: string) : CNF =
    normalize (parse s)

In [4]:
parseKNF "p ∨ r → q"

set [set [Pos "q"; Neg "p"]; set [Pos "q"; Neg "r"]]

## Auxiliary Functions

The function `atMostOneChanging` takes one argument:
- `k` is the number of a crossing, where $k \geq 1$.

It returns a set of clauses that express that at most one element of the set `{ Wolf, Goat, Cabbage }` is on the boat during the $k^{\textrm{th}}$ crossing and, furthermore, this element is with the farmer.  For example, for $k=1$, the following formula is computed and returned as a set of clauses:
```
¬(W1 ↔ W0) → ((G1 ↔ G0) ∧ (C1 ↔ C0) ∧ (F0 ↔ W0))
¬(G1 ↔ G0) → ((C1 ↔ C0) ∧ (W1 ↔ W0) ∧ (F0 ↔ G0))
¬(C1 ↔ C0) → ((W1 ↔ W0) ∧ (G1 ↔ G0) ∧ (F0 ↔ C0))
```
The first line can be read as follows:  If the wolf crosses the river, the goat and the cabbage have to stay put. Furthermore, the farmer has to be on the same side as the wolf initially.

**Implementation:** `L` is the array of the names `W`, `G`, and `C`.  For the element `L[i]`, the other two elements are `L[(i+1) % 3]` and `L[(i+2) % 3]`.

In [5]:
let atMostOneChanging (k: int) : CNF =
    let L = [| "W"; "G"; "C" |]
    Set.unionMany [ for i in 0 .. 2 ->
                      let a, b, c = L[i], L[(i+1) % 3], L[(i+2) % 3]
                      parseKNF (sprintf "¬(%s%d ↔ %s%d) → ((%s%d ↔ %s%d) ∧ (%s%d ↔ %s%d) ∧ (F%d ↔ %s%d))"
                                        a k a (k-1)  b k b (k-1)  c k c (k-1)  (k-1) a (k-1)) ]

The function `notWolfAndGoat` takes one argument:
- `k` is the number of a crossing.

It returns a set of clauses that specifies that the wolf is not with the goat after the $k^{\textrm{th}}$ crossing if the farmer is on the other side. 

In [6]:
let notWolfAndGoat (k: int) : CNF =
    parseKNF (sprintf "¬ F%d → ¬ (W%d ∧ G%d)" k k k)
    + parseKNF (sprintf "F%d → ¬ (¬ W%d ∧ ¬ G%d)" k k k)

The function `notGoatAndCabbage` takes one argument:
- `k` is the number of a crossing.

It returns a set of clauses that specifies that the goat is not with the cabbage after the $k^{\textrm{th}}$ crossing if the farmer is on the other side. 

In [7]:
let notGoatAndCabbage (k: int) : CNF =
    parseKNF (sprintf "¬ F%d → ¬ (G%d ∧ C%d)" k k k)
    + parseKNF (sprintf "F%d → ¬ (¬ G%d ∧ ¬ C%d)" k k k)

The function `allClauses` takes one argument:
- `n` is the number of crossings.

It returns a set of clauses that specify that the problem can be solved in `n` crossings.

In [8]:
let allClauses (n: int) : CNF =
    let mutable Clauses : CNF = Set.empty
    // Start
    Clauses <- Clauses + parseKNF "F0"
    Clauses <- Clauses + parseKNF "W0"
    Clauses <- Clauses + parseKNF "G0"
    Clauses <- Clauses + parseKNF "C0"
    // Goal
    Clauses <- Clauses + parseKNF (sprintf "¬ F%d" n)
    Clauses <- Clauses + parseKNF (sprintf "¬ W%d" n)
    Clauses <- Clauses + parseKNF (sprintf "¬ G%d" n)
    Clauses <- Clauses + parseKNF (sprintf "¬ C%d" n)
    // Crossings
    for k in 1 .. n do
        // The farmer changes sides on every turn.
        Clauses <- Clauses + parseKNF (sprintf "F%d ↔ ¬ F%d" k (k-1))
        Clauses <- Clauses + notWolfAndGoat k
        Clauses <- Clauses + notGoatAndCabbage k
        Clauses <- Clauses + atMostOneChanging k
    Clauses

The function `main` takes no arguments.  It solves the problem by trying all odd numbers of crossings $n = 1, 3, 5, \cdots$ until a solution is found.  Only odd numbers have to be tried, since the farmer changes sides on every crossing and has to end up on the eastern shore.  It returns the pair `(n, Solution)`, where `Solution` is the set of unit clauses describing the solution.

**Implementation:** The search is implemented by the local recursive function `search`, which tries `n` crossings.  If the clauses for `n` crossings are unsatisfiable, `n + 2` crossings are tried next.

In [9]:
let main () : int * CNF =
    let rec search (n: int) : int * CNF =
        printfn "n = %d" n
        let S = solve (allClauses n)
        if S <> set [ Set.empty ] then
            (n, S)
        else
            search (n + 2)
    search 1

Solving the problem takes only a fraction of a second.

In [10]:
#!time
let (n, Solution) = main ()

n = 

1

n = 

3

n = 

5

n = 

7

Wall time: 64.2585ms

In [11]:
Solution

set [set [Pos "C0"]; set [Pos "C1"]; set [Pos "C2"]; set [Pos "C3"]; set [Pos "C4"]; set [Pos "F0"]; set [Pos "F2"]; set [Pos "F4"]; set [Pos "F6"]; set [Pos "G0"]; set [Pos "G4"]; set [Pos "G5"]; set [Pos "G6"]; set [Pos "W0"]; set [Pos "W1"]; set [Pos "W2"]; set [Neg "C5"]; set [Neg "C6"]; set [Neg "C7"]; set [Neg "F1"]; set [Neg "F3"]; set [Neg "F5"]; set [Neg "F7"]; set [Neg "G1"]; set [Neg "G2"]; set [Neg "G3"]; set [Neg "G7"]; set [Neg "W3"]; set [Neg "W4"]; set [Neg "W5"]; set [Neg "W6"]; set [Neg "W7"]]

The function `createDictionary` takes two arguments:
- `Solution` is a set of unit clauses describing a solution,
- `n` is the number of crossings.

It returns a map that assigns to every variable `Fi`, `Wi`, `Gi`, and `Ci` the number `1` if the variable is true and `0` otherwise.

In [12]:
let createDictionary (Solution: CNF) (n: int) : Map<string, int> =
    Map [ for i in 0 .. n do
            for name in [ "F"; "W"; "G"; "C" ] ->
                let v = sprintf "%s%d" name i
                (v, if Solution.Contains (set [ Pos v ]) then 1 else 0) ]

In [13]:
createDictionary Solution n

map [("C0", 1); ("C1", 1); ("C2", 1); ("C3", 1); ("C4", 1); ("C5", 0); ("C6", 0); ("C7", 0); ("F0", 1); ("F1", 0); ("F2", 1); ("F3", 0); ("F4", 1); ("F5", 0); ("F6", 1); ("F7", 0); ("G0", 1); ("G1", 0); ("G2", 0); ("G3", 0); ("G4", 1); ("G5", 1); ("G6", 1); ("G7", 0); ("W0", 1); ("W1", 1); ("W2", 1); ("W3", 0); ("W4", 0); ("W5", 0); ("W6", 0); ("W7", 0)]

The function `showSolution` takes two arguments:
- `Solution` is a map assigning `0` or `1` to the variables, as computed by `createDictionary`,
- `n` is the number of crossings.

It prints the solution.  For every $i \in \{0, \cdots, n\}$, it prints the situation after the $i^{\textrm{th}}$ crossing.  The western shore is printed on the left, the eastern shore on the right.  Between two situations, the boat is shown with the farmer and the object that is transported.

**Implementation:** The function `rep k s` returns the string `s` repeated `k` times.  Hence, `rep F "🧑‍🌾"` is the farmer if `F = 1` and the empty string if `F = 0`.  If the farmer is on the western shore, an object that is on the western shore before the crossing and on the eastern shore after the crossing is on the boat.  Otherwise, it is the other way around.

In [14]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    for i in 0 .. n do
        let F = Solution[sprintf "F%d" i]
        let W = Solution[sprintf "W%d" i]
        let G = Solution[sprintf "G%d" i]
        let C = Solution[sprintf "C%d" i]
        printfn "%s%s%s%s%s%s%s%s%s" (rep F "🧑‍🌾") (rep W "🐺") (rep G "🐐") (rep C "🥦") (String.replicate 28 " ")
                                     (rep (1-F) "🧑‍🌾") (rep (1-W) "🐺") (rep (1-G) "🐐") (rep (1-C) "🥦")
        if i < n then
            let sign = if F = 1 then 1 else -1
            let WB = sign * (W - Solution[sprintf "W%d" (i+1)])
            let GB = sign * (G - Solution[sprintf "G%d" (i+1)])
            let CB = sign * (C - Solution[sprintf "C%d" (i+1)])
            printfn "%s🌊  🧑‍🌾%s%s%s 🌊" (String.replicate 12 " ") (rep WB "🐺") (rep GB "🐐") (rep CB "🥦")

In [15]:
showSolution (createDictionary Solution n) n

🧑‍🌾

🐺

🐐

🥦

🌊  🧑‍🌾

...

 🌊

🧑‍🌾

🐺

🐐

🥦